# Avian Influenza Dataset Pipeline

Author: Alexander Maksiaev

Purpose: Create weekly dataset using NCBI Virus, Andersen Lab, and GISAID.

Notes: 
* This script only works in a Linux environment with bioconda and ncbi_datasets installed. 
* The directory where this script is housed should also house "utils.py". 
* All data from GISAID must be downloaded prior to running this script.

## Housekeeping

In [ ]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import shutil 
import numpy as np
from itertools import islice
from datetime import datetime
from collections import defaultdict 

os.chdir("/data/maksiaevai/Avian_Flu/")
print(os.listdir())
import importlib
import pipeline_funcs
importlib.reload(pipeline_funcs)
from pipeline_funcs import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

['unassigned_trees_prep.ipynb', 'unassigned_check.ipynb', 'archive', 'other_mammals_counts_01_02_2026.ipynb', 'wendy_data.ipynb', 'genotype_host_table.ipynb', 'openapi3.docs.yaml', 'GISAID_Figures.ipynb', 'preprint_gisaid_check.ipynb', 'D1.1_sources.ipynb', 'human_case_by_genotype_lollipop_plot.R', 'pipeline.ipynb', 'usda_cat_list.ipynb', 'genoflu_check.ipynb', '.ipynb_checkpoints', 'B3_13_APR14_r2t.ipynb', 'utils.py', 'north_america_only.ipynb', 'paloma_keep.ipynb', 'bash', 'rejects.ipynb', 'update_metadata_v3.ipynb', 'feline_relabeling.ipynb', 'h5n1_cross_ref.ipynb', 'genoflu.yml', '.venv', 'deduplicator_fasta.ipynb', 'Step3_avian_flu_GISAID_v5.ipynb', 'genoflu.py', 'pipeline_funcs.py', 'concatenator_v2.ipynb', 'Step2_avian_flu_Andersen_v5.ipynb', 'Step1_avian_flu_NCBI_Virus_v5.ipynb', 'Paloma copy 2.ipynb', 'missing_metadata_barchart.ipynb', '__pycache__', 'tree_relabel_keep_bootstrap_aim.py', '11_01_2024_04_01_2025_non_D1_1_D1_3_IN_OH.ipynb', '.git', 'references', 'metadata_maker.i

In [ ]:
# Directory paths and input

# Input
# browser = input("Browser (Firefox, Chrome, or Edge): ")
# sleep_time = input("Seconds to wait in between clicks (recommended 5): ")
# locations = input("Locations (separate with a comma followed by a space in between locations): ")
# start_date = input("Start date (format: MM-DD-YYYY): ")
# end_date = input("End date (format: MM-DD-YYYY): ")
# prev_end_date = input("End date of previous dataset (format: MM-DD-YYYY): ")
# serotype = input("Serotype (e.g. H5N1): ")
# serotypes = list(serotype)
# genotypes = input("Genotypes (separate with commas and no spaces in between genotypes): ")
# genotypes = genotypes.split(",")


# Dates and locations
# browser = "Firefox"
# sleep_time = "6"
locations = "Antarctica, North America, South America"
start_date = "11-01-2021"
end_date = "03-13-2026"
prev_end_date = "03-06-2026"
date_range = start_date + "--" + end_date
# prev_date_range = start_date + "--" + prev_end_date

# Maintenance serotypes and genotypes
serotypes = ["H5N1"]
genotypes = ["B3.13", "D1.1", "D1.3", "Not"]

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/"

# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
# references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"

home = "/data/maksiaevai/"
# downloads = "/home/maksiaevai/Downloads/"
references = "/data/maksiaevai/Avian_Flu/references/"

# downloads_saved = home + "NCBI_Virus/downloads/" + date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 
# prev_downloads_saved = home + prev_date_range + "_" + locations.replace(",", "_").replace(" ", "_") + "/" 


complete_files = home + date_range + "_" + locations.replace(", ", "_").replace(" ", "_") + "/"
andersen = home + "avian-influenza/metadata/"
# ncbi_virus = complete_files + "NCBI_Virus/"
downloads = complete_files + "downloads/"
temp_files = complete_files + "temp_files/"

for folder in [complete_files, downloads, temp_files]:
    if not os.path.exists(complete_files): # checking if the directory exists or not
        os.makedirs(complete_files) # if the directory is not present then create it

os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# All serotypes and genotypes
# serotype = ""
# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])


## Downloading Data

### NCBI Virus

In [ ]:
input("User downloaded NCBI Virus data? (ENTER if yes)")

In [ ]:
os.chdir(downloads)

ncbi_virus_metadata = pd.read_csv("sequences.csv")
print(len(ncbi_virus_metadata))

# Make sure we only have completed sequences -- 8 segments each 

ncbi_virus_metadata_counts = ncbi_virus_metadata.groupby(ncbi_virus_metadata.Isolate, as_index=False).size()
# print(metadata_counts)
ncbi_virus_metadata_counted = ncbi_virus_metadata.merge(ncbi_virus_metadata_counts, on="Isolate")

# Only keep those with size >= 8

ncbi_virus_metadata_complete_segs = ncbi_virus_metadata_counted[ncbi_virus_metadata_counted["size"] >= 8] # May have duplicates
ncbi_virus_metadata_complete_segs = ncbi_virus_metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="last") # Get rid of duplicate segments

# Now only accept == 8 segments

ncbi_virus_metadata_segments = ncbi_virus_metadata_complete_segs[ncbi_virus_metadata_complete_segs["size"] == 8]

ncbi_virus_sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

ncbi_virus_sequences_fasta["Accession"] = ncbi_virus_sequences_fasta["full_header"].apply(lambda x: x.split(" |")[0].replace(">",""))

print(ncbi_virus_sequences_fasta["full_header"])

# Double-check the de-duplication
print(len(ncbi_virus_sequences_fasta)) 
# print(sequences_fasta.head())
print(len(ncbi_virus_metadata_segments))

# Add sequences to the dataframe
ncbi_virus_metadata_segments = pd.merge(ncbi_virus_metadata_segments, ncbi_virus_sequences_fasta, on="Accession") # , "Segment"])

### Andersen

In [6]:
os.chdir(home + "avian-influenza/")

In [ ]:
%%bash
git pull remote set-url https://github.com/andersen-lab/avian-influenza.git

Already up to date.


In [ ]:
os.chdir(andersen)
andersen_metadata = pd.read_csv("SraRunTable_automated_normalized.tsv", delimiter="\t", low_memory=False)
print(len(andersen_metadata)) 
print(andersen_metadata.columns)

# Find the name of the state sample was collected in
andersen_metadata["name_state"] = andersen_metadata["geo_loc_name"].apply(lambda x: x.split("/")[1]) # if len(x.split("/")[1]) > 0 else x.split("/")[0])

# Convert the dates to date format so we can compare
andersen_metadata["ReleaseDate"] = andersen_metadata["ReleaseDate"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y-%m-%d"))
andersen_metadata = andersen_metadata[andersen_metadata["ReleaseDate"] >= dateutil.parser.parse(start_date).strftime("%Y-%m-%d")] # Find only >= last date using Release Date from metadata 
andersen_metadata = andersen_metadata[andersen_metadata["ReleaseDate"] <= dateutil.parser.parse(end_date).strftime("%Y-%m-%d")] # Find only <= update date using Release Date from metadata
andersen_metadata = andersen_metadata[andersen_metadata["is_retracted"] == False]

print(len(andersen_metadata)) 

20686
Index(['Run', 'Assay Type', 'AvgSpotLen', 'Bases', 'BioProject', 'BioSample',
       'BioSampleModel', 'Bytes', 'Center Name', 'Collection_Date', 'Consent',
       'DATASTORE filetype', 'DATASTORE provider', 'DATASTORE region',
       'Experiment', 'geo_loc_name_country', 'geo_loc_name_country_continent',
       'geo_loc_name', 'Host', 'Instrument', 'isolate', 'Library Name',
       'LibraryLayout', 'LibrarySelection', 'LibrarySource', 'Organism',
       'Platform', 'ReleaseDate', 'create_date', 'version', 'Sample Name',
       'SRA Study', 'serotype', 'isolation_source', 'BioSample Accession',
       'is_retracted', 'retraction_detection_date_utc'],
      dtype='object')
20198


### GISAID

In [9]:
input("User downloaded GISAID data? (ENTER if yes)")

''

In [ ]:
gisaid_all_metadata_files = []
gisaid_all_fasta_files = []

for dirpath, dirs, files in os.walk(downloads):
    for file in files:
        file_name = os.path.join(dirpath, file)
        print(file_name)

        # Now go through files and get contents
        if ".xls" and "gisaid" in file_name:
            metadata = pd.read_excel(file_name)
            gisaid_all_metadata_files.append(metadata)
        if ".fasta" and "gisaid" in file_name:
            fasta_file = fasta_df(file_name, states_ref) # Convert fasta file to dataframe
            gisaid_all_fasta_files.append(fasta_file)
    break 

/data/maksiaevai/11-01-2021--02-20-2026_Antarctica_North_America_South_America/downloads/all_alphainfluenza.zip
/data/maksiaevai/11-01-2021--02-20-2026_Antarctica_North_America_South_America/downloads/gisaid_epiflu_sequence.fasta
/data/maksiaevai/11-01-2021--02-20-2026_Antarctica_North_America_South_America/downloads/gisaid_epiflu_sequence_1.fasta
/data/maksiaevai/11-01-2021--02-20-2026_Antarctica_North_America_South_America/downloads/gisaid_epiflu_isolates_1.xls
/data/maksiaevai/11-01-2021--02-20-2026_Antarctica_North_America_South_America/downloads/alphainfluenza-after_11012021.zip
/data/maksiaevai/11-01-2021--02-20-2026_Antarctica_North_America_South_America/downloads/gisaid_epiflu_isolates.xls


In [ ]:
print(gisaid_all_metadata_files[0])

            Isolate_Id                                     PB2 Segment_Id  \
0     EPI_ISL_19792375  EPI4149450|A/canada goose/West Virginia/006699...   
1     EPI_ISL_19792373  EPI4149434|A/canada goose/USA/007062-001/2025_PB2   
2     EPI_ISL_19792372  EPI4149426|A/canada goose/New York/007062-002/...   
3     EPI_ISL_19792364  EPI4149376|A/canada goose/Virginia/007111-001/...   
4     EPI_ISL_19792362    EPI4149360|A/snow goose/USA/007115-001/2025_PB2   
...                ...                                                ...   
9099  EPI_ISL_19775374      EPI4088862|A/Mallard/QC/FAV-1267-102/2022-PB2   
9100  EPI_ISL_19775373      EPI4088854|A/Mallard/QC/FAV-1267-101/2022-PB2   
9101  EPI_ISL_19775372       EPI4088846|A/Mallard/QC/FAV-1267-90/2022-PB2   
9102  EPI_ISL_19775379  EPI4309208|EPI_ISL_19775379|A/American_Black_D...   
9103  EPI_ISL_19775378  EPI4309204|EPI_ISL_19775378|A/American_Black_D...   

                                         PB1 Segment_Id  \
0     EPI4149448

: 

## Clean up dataframes

### NCBI Virus

Find genotypes

In [ ]:
# Create 8 fasta files per segment

# Get all the segments
ncbi_virus_metadata_segments["Partial_Header"] = ncbi_virus_metadata_segments["Assembly"].apply(lambda x: ">" + x if x == x else x) # .apply(lambda x: x.split("|")[-1].split(")")[0]) # Get only genbank name

print(ncbi_virus_metadata_segments["Partial_Header"].values[0:5])

# Create list of dataframes
ncbi_virus_df_list = []
for partial_header in list(set(ncbi_virus_metadata_segments["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = ncbi_virus_metadata_segments[ncbi_virus_metadata_segments["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]:
            df["Partial_Header"] = df["Partial_Header"].apply(lambda x: x.replace(c, "_"))
            
        df["full_header"] = df["Partial_Header"]
        ncbi_virus_df_list.append(df)

print(ncbi_virus_metadata_segments["Partial_Header"])

print(ncbi_virus_df_list[0]["full_header"].values[:10])

# Make fasta files
for df in ncbi_virus_df_list:
    segments = list(set(df["Segment"].apply(lambda x: int(x)).values))
    for segment in segments:
        one_row = df[df["Segment"] == segment]
        df_to_fasta(one_row, str(segment) + "_seg.fasta", temp_files)

In [ ]:
%%bash
source myconda
conda activate genoflu

cd ../GenoFLU-multi

python bin/genoflu-multi.py -f $temp_files

In [ ]:
# Merging

os.chdir(temp_files + "results/")

# Read in genoflu results
output_genoflu = pd.read_csv("results.tsv", delimiter="\t") # This also shows the IDs of the new sequences -- incorporate into sequences report

# Create partial headers to merge genoflu results with previously unknown segments
ncbi_virus_metadata_segments["Partial_Header_Merge"] = ncbi_virus_metadata_segments["Assembly"] 
for c in ["/", "|", " ", ":", ",", "(", ")", "-", "."]: # Forbidden punctuation
    ncbi_virus_metadata_segments["Partial_Header_Merge"] = ncbi_virus_metadata_segments["Partial_Header_Merge"].apply(lambda x: x.replace(c, "_") if x==x else x)

# Reformat so we can merge
output_genoflu["Partial_Header_Merge"] = output_genoflu["Strain"] #.apply(lambda x: re.split(r'Influenza_A_virus__\D*__\D*_', x)[-1])
# output_genoflu["Partial_Header_Merge"] = output_genoflu["Partial_Header_Merge"].apply(lambda x: re.split(r'_H.N._(20|19).{2}_.{2}_.{2}', x)[0])

# Merge
ncbi_virus_metadata_genoflu = ncbi_virus_metadata_segments.merge(output_genoflu, how="inner", on="Partial_Header_Merge") #, suffixes=('_left', '_right')) 

# Fill the rest of the 8 segments with the same genotype
ncbi_virus_metadata_genoflu = ncbi_virus_metadata_genoflu.ffill(limit_area="inside")

print(ncbi_virus_metadata_genoflu)


In [ ]:
# Cut down to only columns we want
ncbi_virus_metadata_genoflu = ncbi_virus_metadata_genoflu[["Accession", "Assembly", "GenBank_Title", "Host", "Collection_Date", "SRA_Accession", "Isolate", "Genotype_official", "Geo_Location", "full_header", "sequence", "Serotype", "Segment", "File Name", "Partial_Header"]] #, "Strain"]]

# Get genbank strain name
ncbi_virus_metadata_genoflu["genbank_name"] = ncbi_virus_metadata_genoflu["GenBank_Title"].apply(lambda x: x.split("(")[1] if x == x else x)
ncbi_virus_metadata_genoflu["Host"] = ncbi_virus_metadata_genoflu["genbank_name"].apply(lambda x: x.split("/")[1] if x == x else x)

ncbi_virus_metadata_genoflu = ncbi_virus_metadata_genoflu.dropna(subset="genbank_name") # [metadata_genoflu["Genotype"]  == "B3.13"]

### Andersen

### GISAID

## Animals

In [ ]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(ncbi_virus_metadata_genoflu)
unique_animals_all.append(sort_animals_andersen(andersen_metadata))
unique_animals_all.append(sort_animals(gisaid))

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


In [ ]:
input("Check animals output. Afterwards, press ENTER to continue.")

## Relabeling Sequences

### NCBI Virus

In [ ]:
# Re-label sequences with no assigned genotype as "Not"

ncbi_virus_metadata_genoflu["Genotype_official"] = ncbi_virus_metadata_genoflu["Genotype_official"].apply(lambda x: "Not" if "Not assigned" in str(x) else x) # Unassigned segments are labeled "Unassigned"
ncbi_virus_metadata_genoflu["Genotype"] = ncbi_virus_metadata_genoflu["Genotype_official"] # Rename column again to not break old code

In [ ]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
ncbi_virus_metadata_genoflu = ncbi_virus_metadata_genoflu.fillna("")
ncbi_virus_metadata_genoflu["Host"] = ncbi_virus_metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(ncbi_virus_metadata_genoflu, animals_ref)

# Get the years
ncbi_virus_metadata_genoflu["Years"] = ncbi_virus_metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# Get geographic locations
ncbi_virus_metadata_genoflu["Geo_Location_Abrv"] = ncbi_virus_metadata_genoflu["Geo_Location"].apply(lambda x: 
                                                                        # If "x" has the state abbreviation (e.g. "MD")
                                                                        states_ref.loc[states_ref["Abbreviation"].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0] 
                                                                        + "-" + 
                                                                        x.split(" ")[-1]
                                                                        if states_ref["Abbreviation"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has the full state name (e.g. "Maryland")
                                                                        else states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Country'].iloc[0]
                                                                        + "-" + 
                                                                        states_ref.loc[states_ref['State'].str.contains('|'.join(re.sub(":? ", ",", x).replace(" ", "_").split(',')), regex=True), 'Abbreviation'].iloc[0] 
                                                                        if states_ref["State"].str.contains("|".join((re.sub(":? ", ",", x).replace(" ", "_").split(','))), regex=True).any()
                                                                        # If "x" has neither the state abbreviation nor the full state name nor is "USA"
                                                                        else 
                                                                        x
                                                                        )

ncbi_virus_metadata_genoflu["Geo_Location_Abrv"] = ncbi_virus_metadata_genoflu["Geo_Location_Abrv"].apply(lambda x: x.split("-")[0] if x.split("-")[-1] == "" or x.split("-")[-1] == x.split("-")[0] else x)


print(ncbi_virus_metadata_genoflu["Geo_Location_Abrv"])

In [ ]:
# If there is no SRA Accession, replace identifier with Accession
ncbi_virus_metadata_genoflu["SRA_Accession"] = np.where(ncbi_virus_metadata_genoflu['SRA_Accession'] == "", ncbi_virus_metadata_genoflu['Assembly'].apply(lambda x: x.split(".")[0]), ncbi_virus_metadata_genoflu['SRA_Accession'])
# If there is no Assembly, replace identifier with Accession -- only for PB2
ncbi_virus_metadata_genoflu["Identifier"] = np.where(ncbi_virus_metadata_genoflu["SRA_Accession"] == "", ncbi_virus_metadata_genoflu["Accession"].apply(lambda x: ncbi_virus_metadata_genoflu[ncbi_virus_metadata_genoflu["Accession"] == x] if ncbi_virus_metadata_genoflu[ncbi_virus_metadata_genoflu["Accession"] == x].loc[:, "Segment"].values[0] == 1 else np.nan), ncbi_virus_metadata_genoflu["SRA_Accession"])
# Fill in other nans with PB2 Accession (interpolate, maximum of 7 other sequences)
ncbi_virus_metadata_genoflu.loc[:,"Identifier"] = ncbi_virus_metadata_genoflu.loc[:, "Identifier"].ffill(limit=7, limit_area="inside")

print(ncbi_virus_metadata_genoflu)

In [ ]:

# Make new labels
names = ">" + ncbi_virus_metadata_genoflu["Identifier"].astype(str) + "|" + ncbi_virus_metadata_genoflu["genbank_name"] + "|" + ncbi_virus_metadata_genoflu["Serotype"] + "|" + ncbi_virus_metadata_genoflu["Geo_Location_Abrv"] + "|" + ncbi_virus_metadata_genoflu["Collection_Date"].astype(str) + "|" + ncbi_virus_metadata_genoflu["Host_Type"] + "|" + ncbi_virus_metadata_genoflu["Genotype"]

ncbi_virus_metadata_genoflu["Name"] = names

### Andersen

### GISAID

## Make complete FASTA files

### NCBI Virus

In [ ]:
# Set up segments

# if len(genotypes) > 3: # If we're not doing maintenance only
#     genotypes.append("Not assigned") # Make sure unassigned genotypes are included
segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"} # Name segments 
ncbi_virus_metadata_genoflu["Segment_Name"] = ncbi_virus_metadata_genoflu["Segment"].apply(lambda x: int(x)).map(segments)

# Separate into several dataframes based on genotype + segment
ncbi_virus_segment_genotype_dfs = []
for segment in segments.values():
    m_g = ncbi_virus_metadata_genoflu[ncbi_virus_metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype"] == genotype)] 
            # df.drop_duplicates(subset="SRA_Accession", keep="first", inplace=True)
            ncbi_virus_segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

In [ ]:
# Create FASTA files

os.chdir(complete_files)

for df in ncbi_virus_segment_genotype_dfs:
    print(df)
    
    df = df.reset_index()
    if len(df["Genotype"].values[0]) > 0: # If we have assigned genotypes, including "Unassigned"
        file_name = "NCBI_Virus_" + df["Genotype"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            # names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        
    output_file.close()

### Andersen

### GISAID

## Merging

## Final FASTAs

## New sequences report